# UL-UNAS Denoising & Silero VAD Pipeline

This notebook demonstrates an end-to-end audio processing and Voice Activity Detection (VAD) pipeline designed for lightweight deployment environments (such as Apache Beam / Google Cloud Dataflow).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/segmentation/silero_and_ul_unas_wet_dry_VAD_mixture.ipynb)

### Key Features:
* **Zero PyTorch Dependency**: To avoid massive dependency bloat in worker instances, this pipeline relies exclusively on **ONNX Runtime**, **NumPy**, and **Pedalboard**. Resampling heavily relies on a custom NumPy implementation of `torchaudio`'s sinc resampler to maintain the precise audio characteristics required by the denoiser.
* **UL-UNAS Denoiser**: Applies a [state-of-the-art streaming denoiser](https://github.com/Xiaobin-Rong/ul-unas) to remove background noise.
* **Wet/Dry Audio Mixture**: Blends the output of the deep-learning denoiser ("wet" signal) with the original bandpass-filtered audio ("dry" signal) controlled by an adjustable `blend_ratio`. Retaining a portion of the original bandpassed voice helps mitigate over-gating effects, voice muffling, or premature trailing-syllable truncation, which optimizes downstream VAD accuracy on degraded, noisy radio tracks.
* **Silero VAD**: Detects speech segments using ONNX-based Silero VAD, enhanced with custom hysteresis (distinct onset/offset thresholds) and padding logic to prevent fragmented segments.
* **Audio Mastering**: Incorporates dry/wet blending, bandpass filtering, and an EQ presence boost to optimize voice clarity before VAD processing.
* **Evaluation**: Includes built-in tools to visualize audio waveforms, VAD probabilities, and automatically evaluate VAD recall/precision against labeled ground-truth segments.

**Note**: Currently, this Colab has evaluation support configured *only* for the specific audio files listed as keys in the `GROUND_TRUTH_SEGMENTS` dictionary. These files are located in the `models/colabs/data/segmentation` directory.

In [ ]:
%pip install -q onnxruntime pedalboard

In [ ]:
# @title Constants
SILERO_MODEL_FILENAME = "silero_vad.onnx"
SILERO_VERSION = "6.2.1"
ULUNAS_MODEL_FILENAME = "ulunas_stream_simple.onnx"

# Consolidated DSP Constants
TARGET_SAMPLE_RATE = 16000
VAD_CHUNK_SIZE = 512
VAD_CONTEXT_SIZE = 64
VAD_STATE_SIZE = 128
VAD_NOISE_LEVEL = 0.002
VAD_DEFAULT_SEED = 2147483647
INT16_MAX_FLOAT = 32768.0

In [ ]:
# @title Download the Silero VAD model
!wget -q -N --show-progress -N https://raw.githubusercontent.com/snakers4/silero-vad/v{SILERO_VERSION}/src/silero_vad/data/{SILERO_MODEL_FILENAME}

In [ ]:
# @title Download the UL-UNAS denoiser model
!wget -q -N --show-progress -N https://raw.githubusercontent.com/Xiaobin-Rong/ul-unas/main/ulunas_onnx/onnx_models/{ULUNAS_MODEL_FILENAME}

In [ ]:
# @title Imports
import base64
import io
import math
import os
import pathlib
import subprocess
import tempfile
import wave

from google.colab import files
from IPython.display import Audio, HTML, display
import matplotlib.pyplot as plt
from numba import njit
import numpy as np
import onnxruntime as ort
from pedalboard import (
    Compressor,
    HighpassFilter,
    LowpassFilter,
    PeakFilter,
    Pedalboard,
)
import soundfile as sf

In [ ]:
# @title Constants

GROUND_TRUTH_SEGMENTS = {
    "test_stress.flac": [(0.4, 2.85)],
    "test_joined.flac": [(8.3, 10.7), (12.3, 15.6), (20.3, 23.0), (26.2, 27.0)],
    "test_bcfy.flac": [(0.0, 1.1), (1.95, 5.3), (7.25, 10.9), (11.6, 12.2)],
    "test_dispatch_amador.flac": [
        (2.7, 12.5),
        (14.4, 15.8),
        (17.5, 24.6),
        (27.3, 29.7),
        (31.4, 33.7),
        (38.1, 40.5),
        (47.2, 49.4),
        (56.2, 60.6),
        (62.6, 65.3),
    ],
    "test_dispatch_sku.flac": [
        (0.420, 2.593),
        (3.3, 5.788),
        (6.242, 8.838),
        (8.861, 11.044),
        (11.691, 14.717),
        (14.811, 17.014),
        (17.781, 19.707),
        (20.253, 22.040),
        (22.843, 24.669),
        (25.547, 27.728),
        (28.471, 29.830),
        (30.845, 32.907),
        (33.003, 34.615),
        (35.704, 37.877),
        (40.570, 41.772),
        (42.467, 44.470),
        (45.874, 49.212),
        (49.373, 51.884),
        (52.768, 54.178),
    ],
    "test_middlebury_quiet_segments.mp3": [(0.47, 1.4), (3.9, 6.4)],
    "test_quiet_speech_loud_transient.mp3": [(0.213, 0.8), (2.037, 3.869)],
    "test_middlebury_quiet_spiky.mp3": [(0.18, 1.45)],
    "test_only_static_middlebury.mp3": [],
    "test_tone_only.flac": [],
}

In [ ]:
# @title Helper classes/functions

# Note: This Colab intentionally avoids using Pytorch to avoid the dependency
# bloat that would result in Dataflow worker instances. The performance
# of the UL-UNAS denoiser is highly dependent on the specific resampling
# implementation in torchaudio, so we provide an equivalent here in numpy.


def get_periodic_hann(window_length: int) -> np.ndarray:
    """Generates a periodic Hann window of length `window_length`."""
    return 0.5 * (
        1 - np.cos(2 * np.pi * np.arange(window_length) / window_length)
    )


def custom_numpy_stft(
    wav: np.ndarray, n_fft: int = 512, hop_length: int = 256
) -> np.ndarray:
    """Calculates Short-Time Fourier Transform (STFT) via NumPy arrays.

    Args:
        wav: Input 1D or 2D NumPy array of the source audio.
        n_fft: Number of fast Fourier Transform points.
        hop_length: Step size for shifting the window.

    Returns:
        A stacked NumPy array representing the real/imaginary components.
    """
    window = get_periodic_hann(n_fft)
    pad_len = n_fft // 2
    wav_padded = np.pad(wav, ((0, 0), (pad_len, pad_len)), mode="reflect")
    num_frames = 1 + (wav_padded.shape[1] - n_fft) // hop_length
    strides = (
        wav_padded.strides[0],
        wav_padded.strides[1] * hop_length,
        wav_padded.strides[1],
    )
    frames = np.lib.stride_tricks.as_strided(
        wav_padded, shape=(wav.shape[0], num_frames, n_fft), strides=strides
    )
    windowed = frames * window
    spec = np.fft.rfft(windowed, n=n_fft, axis=-1)
    spec = np.transpose(spec, (0, 2, 1))
    return np.stack((np.real(spec), np.imag(spec)), axis=-1).astype(np.float32)


@njit(fastmath=True)
def _ola_step(
    frames, window, window_sq, out, window_sum, num_frames, hop_length, n_fft
):
    """Accelerates the overlap-add operation for the inverse STFT execution via Numba."""
    batch_size = frames.shape[0]
    for b in range(batch_size):
        for i in range(num_frames):
            start = i * hop_length
            for j in range(n_fft):
                out[b, start + j] += frames[b, i, j] * window[j]
                window_sum[b, start + j] += window_sq[j]


def custom_numpy_istft(
    spec_stacked: np.ndarray,
    length: int,
    n_fft: int = 512,
    hop_length: int = 256,
) -> np.ndarray:
    """Calculates the inverse Short-Time Fourier Transform (ISTFT) using Overlap-Add (OLA)."""
    window = get_periodic_hann(n_fft).astype(np.float32)
    spec_complex = spec_stacked[..., 0] + 1j * spec_stacked[..., 1]
    spec_complex = np.transpose(spec_complex, (0, 2, 1))
    frames = np.fft.irfft(spec_complex, n=n_fft, axis=-1).astype(np.float32)
    batch_size, num_frames, _ = frames.shape
    expected_len = (num_frames - 1) * hop_length + n_fft
    out = np.zeros((batch_size, expected_len), dtype=np.float32)
    window_sum = np.zeros((batch_size, expected_len), dtype=np.float32)
    window_sq = (window**2).astype(np.float32)

    _ola_step(
        frames,
        window,
        window_sq,
        out,
        window_sum,
        num_frames,
        hop_length,
        n_fft,
    )

    window_sum[window_sum < 1e-10] = 1.0
    out = out / window_sum
    pad_len = n_fft // 2
    return out[:, pad_len : pad_len + length]


class TorchaudioHannResampler:
    """Resampling implementation equivalent to Torchaudio's Hann-window sinc method."""

    def __init__(
        self,
        orig_freq: int,
        new_freq: int,
        lowpass_filter_width: int = 6,
        rolloff: float = 0.99,
    ):
        self.orig_freq = orig_freq
        self.new_freq = new_freq
        self.lowpass_filter_width = lowpass_filter_width
        self.gcd = math.gcd(orig_freq, new_freq)
        self.down = orig_freq // self.gcd
        self.up = new_freq // self.gcd
        self.kernel, self.width = self._get_sinc_resample_kernel(
            self.down, self.up, lowpass_filter_width, rolloff
        )

    def _get_sinc_resample_kernel(
        self, orig_freq, new_freq, lowpass_filter_width, rolloff
    ):
        base_freq = min(orig_freq, new_freq) * rolloff
        width = math.ceil(lowpass_filter_width * orig_freq / base_freq)
        idx = (
            np.arange(-width, width + orig_freq, dtype=np.float64)[
                None, None, :
            ]
            / orig_freq
        )
        t = (
            np.arange(0, -new_freq, -1, dtype=np.float64)[:, None, None]
            / new_freq
            + idx
        )
        t *= base_freq
        t = np.clip(t, -lowpass_filter_width, lowpass_filter_width)
        window = np.cos(t * math.pi / lowpass_filter_width / 2) ** 2
        t_pi = t * math.pi
        scale = base_freq / orig_freq
        kernels = np.zeros_like(t_pi)
        mask = t_pi == 0
        kernels[mask] = 1.0
        kernels[~mask] = np.sin(t_pi[~mask]) / t_pi[~mask]
        kernels *= window * scale
        return kernels.astype(np.float64), width

    def resample(self, waveform: np.ndarray) -> np.ndarray:
        is_1d = waveform.ndim == 1
        if is_1d:
            waveform = waveform[np.newaxis, :]
        orig_len = waveform.shape[-1]
        wav_padded = np.pad(
            waveform.astype(np.float64),
            ((0, 0), (self.width, self.width + self.down)),
            mode="constant",
        )
        batch, length = wav_padded.shape
        kernel_size = self.kernel.shape[2]
        num_frames = (length - kernel_size) // self.down + 1
        strides = (
            wav_padded.strides[0],
            wav_padded.strides[1] * self.down,
            wav_padded.strides[1],
        )
        frames = np.lib.stride_tricks.as_strided(
            wav_padded, shape=(batch, num_frames, kernel_size), strides=strides
        )
        resampled = np.tensordot(frames, self.kernel[:, 0, :], axes=([2], [1]))
        resampled = resampled.reshape(batch, -1)
        target_length = int(math.ceil(self.up * orig_len / self.down))
        resampled = resampled[:, :target_length]
        if is_1d:
            return resampled[0].astype(np.float32)
        return resampled.astype(np.float32)


def load_and_resample_audio(
    audio_filepath: str, target_sr: int = TARGET_SAMPLE_RATE
) -> np.ndarray:
    """Loads file via soundfile with fallback logic referencing standard FFmpeg decoders."""
    # If it is an MP3 file, bypass soundfile entirely to prevent silent libsndfile truncation
    is_mp3 = str(audio_filepath).lower().endswith(".mp3")

    if not is_mp3:
        try:
            # Attempt to read normally via soundfile
            audio_data, orig_sr = sf.read(audio_filepath, always_2d=True)
            audio_data = audio_data.mean(axis=1).astype(
                np.float32
            )  # downmix to mono
            use_fallback = False
        except Exception as e:
            print(
                f"[load_and_resample_audio] soundfile failed ({e}), falling back to ffmpeg..."
            )
            use_fallback = True
    else:
        use_fallback = True

    if use_fallback:
        # Get original sample rate using ffprobe
        probe_cmd = [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=sample_rate",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            audio_filepath,
        ]
        orig_sr = int(
            subprocess.check_output(probe_cmd)
            .decode("utf-8")
            .strip()
            .split("\n")[0]
        )

        # Decode audio to raw PCM float32 at its ORIGINAL sample rate
        command = [
            "ffmpeg",
            "-i",
            audio_filepath,
            "-f",
            "f32le",
            "-acodec",
            "pcm_f32le",
            "-ac",
            "1",  # downmix to mono
            "-",
        ]
        pipe = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
        )
        out, _ = pipe.communicate()
        audio_data = np.frombuffer(out, dtype=np.float32)

    # Apply resampler if necessary
    if orig_sr != target_sr:
        print(
            f"[load_and_resample_audio] Resampling from {orig_sr} Hz to {target_sr} Hz using TorchaudioHannResampler..."
        )
        resampler = TorchaudioHannResampler(orig_sr, target_sr)
        audio_data = resampler.resample(audio_data)

    return audio_data


def run_ulunas_denoiser(
    audio_array: np.ndarray,
    model_filename: str,
    n_fft: int = 512,
    hop_length: int = 256,
) -> np.ndarray:
    """Denoises input array audio via an ONNXRuntime inference engine instance."""
    sess = ort.InferenceSession(model_filename)
    inputs = sess.get_inputs()
    outputs = sess.get_outputs()
    audio_input_name = inputs[0].name

    # Initialize states dynamically based on the model's signature
    states = {}
    for inp in inputs[1:]:
        shape = [s if isinstance(s, int) else 1 for s in inp.shape]
        states[inp.name] = np.zeros(shape, dtype=np.float32)

    bp_audio_batched = np.expand_dims(audio_array, axis=0)
    stft_features = custom_numpy_stft(
        bp_audio_batched, n_fft=n_fft, hop_length=hop_length
    )

    num_frames = stft_features.shape[2]
    out_stft = np.zeros_like(stft_features)

    for i in range(num_frames):
        frame = stft_features[:, :, i : i + 1, :]
        ort_inputs = {audio_input_name: frame, **states}
        ort_outs = sess.run(None, ort_inputs)
        out_stft[:, :, i : i + 1, :] = ort_outs[0]
        for j in range(1, len(outputs)):
            states[inputs[j].name] = ort_outs[j]

    return custom_numpy_istft(
        out_stft,
        length=audio_array.shape[0],
        n_fft=n_fft,
        hop_length=hop_length,
    )[0]


def is_tone_segment(sig: np.ndarray) -> bool:
    """Analyzes a candidate audio segment to determine if it is primarily an alert or paging tone."""
    frame_len = 1024
    hop_len = 512
    if len(sig) < frame_len:
        return False

    num_frames = 1 + (len(sig) - frame_len) // hop_len
    shape = (num_frames, frame_len)
    strides = (sig.strides[0] * hop_len, sig.strides[0])
    frames = np.lib.stride_tricks.as_strided(
        sig, shape=shape, strides=strides
    )

    window = 0.5 * (
        1.0 - np.cos(2 * np.pi * np.arange(frame_len) / frame_len)
    )
    windowed = frames * window.astype(np.float32)
    specs = np.abs(np.fft.rfft(windowed, axis=-1)) ** 2

    frame_powers = np.sum(specs, axis=-1)
    max_power = np.max(frame_powers)
    if max_power < 1e-10:
        return False

    active_indices = np.where(frame_powers >= 0.01 * max_power)[0]
    if len(active_indices) == 0:
        return False

    tone_frames = 0
    for idx in active_indices:
        spec = specs[idx]
        total_p = frame_powers[idx]

        # Strongest peak
        k1 = int(np.argmax(spec))
        low1 = max(0, k1 - 3)
        high1 = min(len(spec), k1 + 4)
        p1 = float(np.sum(spec[low1:high1]))

        # Second strongest peak outside first neighborhood
        spec_rem = spec.copy()
        spec_rem[low1:high1] = 0.0
        k2 = int(np.argmax(spec_rem))
        low2 = max(0, k2 - 3)
        high2 = min(len(spec), k2 + 4)
        p2 = float(np.sum(spec[low2:high2]))

        # Third strongest peak outside first two neighborhoods
        spec_rem[low2:high2] = 0.0
        k3 = int(np.argmax(spec_rem))
        low3 = max(0, k3 - 3)
        high3 = min(len(spec), k3 + 4)
        p3 = float(np.sum(spec[low3:high3]))

        if (p1 + p2 + p3) / total_p > 0.85:
            tone_frames += 1

    return (tone_frames / len(active_indices)) >= 0.75


def is_speech_segment(sig: np.ndarray, chunk_size: int) -> bool:
    """Applies dynamic range/spikiness heuristics to reject transient static clicks and quiet noise."""
    if len(sig) == 0:
        return False

    # Compute RMS in chunk-sized windows
    seg_rms = []
    for w_start in range(0, len(sig), chunk_size):
        window = sig[w_start : w_start + chunk_size]
        if len(window) < chunk_size:
            window = np.pad(window, (0, chunk_size - len(window)))
        seg_rms.append(np.sqrt(np.mean(window**2)))

    seg_rms = np.array(seg_rms)
    mean_rms = np.mean(seg_rms)
    median_rms = np.median(seg_rms)
    rms_ratio = mean_rms / median_rms if median_rms > 1e-5 else 999.0

    # 1. Ratio check: reject if there are high spikes with very quiet median (clicks/transients)
    if rms_ratio > 15.0:
        spec = np.abs(np.fft.rfft(sig)) ** 2
        spec = np.maximum(spec[1:], 1e-10)
        a_mean = np.mean(spec)
        g_mean = np.exp(np.mean(np.log(spec)))
        flatness = float(g_mean / a_mean) if a_mean > 1e-10 else 1.0
        if flatness < 0.0005:
            return False

    # 2. Floor check: reject if the segment is extremely quiet
    if mean_rms < 0.001:
        return False

    return True


def extract_vad_segments(
    audio_array: np.ndarray,
    vad_session,
    sr_tensor,
    target_sr: int = TARGET_SAMPLE_RATE,
    chunk_size: int = VAD_CHUNK_SIZE,
    context_size: int = VAD_CONTEXT_SIZE,
    threshold=None,
    threshold_onset: float = 0.5,
    threshold_offset: float = 0.35,
    min_speech_duration_ms: int = 200,
    min_silence_duration_ms: int = 500,
) -> list:
    """Segments audio streams with hysteresis logic applied based on Silero VAD likelihoods."""
    if threshold is not None:
        threshold_onset = threshold
        threshold_offset = threshold

    state = np.zeros((2, 1, VAD_STATE_SIZE), dtype=np.float32)
    context = np.zeros(context_size, dtype=np.float32)

    min_speech_frames = int(
        (min_speech_duration_ms * target_sr / 1000) / chunk_size
    )
    min_silence_frames = int(
        (min_silence_duration_ms * target_sr / 1000) / chunk_size
    )

    triggered = False
    temp_end = 0
    current_speech = {}
    raw_segments = []

    for i in range(0, len(audio_array), chunk_size):
        chunk = audio_array[i : i + chunk_size]
        if len(chunk) < chunk_size:
            chunk = np.pad(chunk, (0, chunk_size - len(chunk)))

        x_with_context = np.concatenate([context, chunk])
        ort_inputs = {
            "input": x_with_context.reshape(
                1, chunk_size + context_size
            ).astype(np.float32),
            "state": state,
            "sr": sr_tensor,
        }

        outputs = vad_session.run(None, ort_inputs)
        prob = float(outputs[0].flatten()[0])
        state = outputs[1]
        context = x_with_context[-context_size:]

        current_frame = i / chunk_size

        if not triggered:
            if prob >= threshold_onset:
                triggered = True
                current_speech["start"] = current_frame
                temp_end = 0
        elif prob < threshold_offset:
            temp_end += 1
            if temp_end >= min_silence_frames:
                current_speech["end"] = current_frame - temp_end
                if (
                    current_speech["end"] - current_speech["start"]
                ) >= min_speech_frames:
                    start_sec = current_speech["start"] * chunk_size / target_sr
                    end_sec = current_speech["end"] * chunk_size / target_sr
                    start_idx = int(start_sec * target_sr)
                    end_idx = int(end_sec * target_sr)
                    seg_signal = audio_array[start_idx:end_idx]
                    if is_speech_segment(
                        seg_signal, chunk_size
                    ) and not is_tone_segment(seg_signal):
                        raw_segments.append((start_sec, end_sec))
                triggered = False
                temp_end = 0
                current_speech = {}
        else:
            temp_end = 0

    if triggered:
        current_speech["end"] = len(audio_array) / chunk_size
        if (
            current_speech["end"] - current_speech["start"]
        ) >= min_speech_frames:
            start_sec = current_speech["start"] * chunk_size / target_sr
            end_sec = current_speech["end"] * chunk_size / target_sr
            start_idx = int(start_sec * target_sr)
            end_idx = int(end_sec * target_sr)
            seg_signal = audio_array[start_idx:end_idx]
            if is_speech_segment(
                seg_signal, chunk_size
            ) and not is_tone_segment(seg_signal):
                raw_segments.append((start_sec, end_sec))
    return raw_segments


def pad_and_merge_segments(
    raw_segments: list, audio_len_sec: float, pad_sec: float = 0.5
) -> list:
    """Adds safety buffers around voice activity segments and resolves overlaps."""
    padded_segments = []
    for start, end in raw_segments:
        p_start = max(0.0, start - pad_sec)
        p_end = min(audio_len_sec, end + pad_sec)
        if padded_segments and padded_segments[-1][1] >= p_start:
            padded_segments[-1] = (
                padded_segments[-1][0],
                max(padded_segments[-1][1], p_end),
            )
        else:
            padded_segments.append((p_start, p_end))
    return padded_segments


def calculate_overlap(seg1: tuple, seg2: tuple) -> float:
    start = max(seg1[0], seg2[0])
    end = min(seg1[1], seg2[1])
    return max(0.0, end - start)


def evaluate_vad_frame_based(
    labeled_segments: list,
    final_segments: list,
    audio_len_sec: float,
    resolution_ms: int = 10,
) -> None:
    """Calculates precision, recall, and overall F1 benchmarks on a frame basis (default 10ms bins)."""
    num_frames = int(np.ceil(audio_len_sec * 1000 / resolution_ms))

    gt_array = np.zeros(num_frames, dtype=bool)
    for start, end in labeled_segments:
        start_frame = int(start * 1000 / resolution_ms)
        end_frame = int(end * 1000 / resolution_ms)
        gt_array[start_frame:end_frame] = True

    det_array = np.zeros(num_frames, dtype=bool)
    for start, end in final_segments:
        start_frame = int(start * 1000 / resolution_ms)
        end_frame = int(end * 1000 / resolution_ms)
        det_array[start_frame:end_frame] = True

    tp = np.sum(gt_array & det_array)
    fp = np.sum(~gt_array & det_array)
    fn = np.sum(gt_array & ~det_array)
    tn = np.sum(~gt_array & ~det_array)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    mr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    print("--- Frame-based Evaluation (10ms resolution) ---")
    print(f"Precision:              {precision * 100:.1f}%")
    print(f"Recall (Coverage):      {recall * 100:.1f}%")
    print(f"F1-Score:               {f1 * 100:.1f}%")
    print(f"False Alarm Rate (FAR): {far * 100:.1f}%")
    print(f"Miss Rate (MR):         {mr * 100:.1f}%\n")


def evaluate_vad_performance(
    labeled_segments: list, final_segments: list
) -> None:
    """Compares target pipeline outcomes against ground truth benchmarks."""
    total_gt_duration = sum(end - start for start, end in labeled_segments)
    total_det_duration = sum(end - start for start, end in final_segments)
    total_overlap = 0.0

    matched_det = set()

    for i, gt_seg in enumerate(labeled_segments):
        print(f"Ground Truth {i} ({gt_seg[0]:.2f}s - {gt_seg[1]:.2f}s):")
        seg_overlap = 0.0
        for j, det_seg in enumerate(final_segments):
            overlap = calculate_overlap(gt_seg, det_seg)
            if overlap > 0:
                matched_det.add(j)
                diff_start = det_seg[0] - gt_seg[0]
                diff_end = det_seg[1] - gt_seg[1]

                start_str = (
                    f"{abs(diff_start):.2f}s {'late' if diff_start > 0 else 'early'}"
                    if diff_start != 0
                    else "exact"
                )
                end_str = (
                    f"{abs(diff_end):.2f}s {'late' if diff_end > 0 else 'early'}"
                    if diff_end != 0
                    else "exact"
                )

                print(
                    f"  -> Overlaps with Detected {j} ({det_seg[0]:.2f}s - {det_seg[1]:.2f}s)"
                )
                print(f"     Difference: Started {start_str}, Ended {end_str}")

                seg_overlap += overlap
                total_overlap += overlap

        gt_len = gt_seg[1] - gt_seg[0]
        print(
            f"  Coverage: {(seg_overlap / gt_len) * 100:.1f}% of this segment detected.\n"
        )

    # Check for False Positives
    unmatched = [j for j in range(len(final_segments)) if j not in matched_det]
    if unmatched:
        print("--- False Positives (Detected but no Ground Truth) ---")
        for j in unmatched:
            print(
                f"  Detected {j}: {final_segments[j][0]:.2f}s - {final_segments[j][1]:.2f}s"
            )
        print()

    recall = (
        (total_overlap / total_gt_duration) * 100
        if total_gt_duration > 0
        else 0.0
    )
    precision = (
        (total_overlap / total_det_duration) * 100
        if total_det_duration > 0
        else 0.0
    )

    print(
        f"Overall Recall/Coverage: {recall:.1f}% of actual speech was detected."
    )
    print(
        f"Overall Precision: {precision:.1f}% of detected speech was actual speech.\n"
    )


def visualize_comparison_with_segments(
    audio_array: np.ndarray,
    labeled_segments: list,
    final_segments: list,
    target_sr: int = TARGET_SAMPLE_RATE,
    container_id: str = "comparison_waveform",
) -> None:
    """Presents comparison waveforms matching pipeline output boundaries to Ground Truth markers."""
    wav_io = io.BytesIO()
    with wave.open(wav_io, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(target_sr)
        wav_file.writeframes(
            (
                np.clip(
                    audio_array * INT16_MAX_FLOAT,
                    -INT16_MAX_FLOAT,
                    INT16_MAX_FLOAT - 1,
                )
            )
            .astype(np.int16)
            .tobytes()
        )
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    regions_js = ""
    # Add Ground Truth regions (Green)
    for i, (s, e) in enumerate(labeled_segments):
        regions_js += f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'GT {i}', color: 'rgba(50, 205, 50, 0.4)'}});\n"

    # Add Detected regions (Purple)
    for i, (s, e) in enumerate(final_segments):
        regions_js += f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'Det {i}', color: 'rgba(138, 43, 226, 0.4)'}});\n"

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>Comparison: Ground Truth vs Detected</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: 'tomato',
            progressColor: 'firebrick',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))


def visualize_audio_with_segments(
    audio_array: np.ndarray,
    segments: list,
    target_sr: int = TARGET_SAMPLE_RATE,
    container_id: str = "waveform",
    wave_color: str = "tomato",
    progress_color: str = "firebrick",
    region_color: str = "rgba(255, 99, 71, 0.4)",
    title_text: str = "Audio Configuration",
) -> None:
    """Creates interactable waveform charts representing slice regions natively via WaveSurfer.js."""
    wav_io = io.BytesIO()
    with wave.open(wav_io, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(target_sr)
        wav_file.writeframes(
            (
                np.clip(
                    audio_array * INT16_MAX_FLOAT,
                    -INT16_MAX_FLOAT,
                    INT16_MAX_FLOAT - 1,
                )
            )
            .astype(np.int16)
            .tobytes()
        )
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    regions_js = "".join(
        [
            f"wsRegions.addRegion({{start: {s}, end: {e}, content: 'Seg {i}', color: '{region_color}'}});\n"
            for i, (s, e) in enumerate(segments)
        ]
    )

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>{title_text} ({len(segments)} segments)</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: '{wave_color}',
            progressColor: '{progress_color}',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))

In [ ]:
# @title Upload an Audio File

print("Please select an audio file to upload:\n")

# Create a fresh temporary directory
temp_dir = tempfile.mkdtemp()

uploaded = files.upload(target_dir=temp_dir)

if uploaded:
    # Grab the filename of the uploaded file
    filename = list(uploaded.keys())[0]
    # The file is saved inside the temp_dir
    AUDIO_FILEPATH = os.path.join(temp_dir, filename)
    print(f"\nSuccess! AUDIO_FILEPATH is now set to: {AUDIO_FILEPATH}")
else:
    AUDIO_FILEPATH = None
    print("\nNo file uploaded.")

In [ ]:
# @title Bandpass + UL-UNAS (Dry/Wet) + EQ + VAD
# @markdown ### Preamble Settings (primes VAD)
preamble_duration_sec = 6  # @param {type:"slider", min:0.0, max:30.0, step:1.0}

# @markdown ### Bandpass Settings
highpass_hz = 300  # @param {type:"slider", min:50, max:1000, step:50}
lowpass_hz = 4000  # @param {type:"slider", min:2000, max:8000, step:100}

# @markdown ### Adaptive AGC Compressor Settings
use_compressor = "Auto"  # @param ["Auto", "True", "False"]
comp_peak_threshold = 0.55  # @param {type:"slider", min:0, max:1, step:0.05}
comp_threshold_db = -15  # @param {type:"slider", min:-60, max:0, step:1}
comp_ratio = 3.0  # @param {type:"slider", min:1.0, max:20.0, step:0.5}
comp_attack_ms = 2.0  # @param {type:"slider", min:0.1, max:50.0, step:0.5}
comp_release_ms = 150  # @param {type:"slider", min:10, max:1000, step:10}

# @markdown ### Denoiser Settings
blend_ratio = 0.8  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

# @markdown ### EQ / Presence Boost Settings
boost_freq_hz = 2500  # @param {type:"slider", min:1000, max:4000, step:100}
boost_gain_db = 10  # @param {type:"slider", min:0.0, max:24.0, step:1.0}
peak_filter_q = 1.0  # @param {type:"slider", min:0.1, max:5.0, step:0.1}

# @markdown ### VAD (Voice Activity Detection) Settings
# fmt: off
vad_threshold_onset = 0.35  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
vad_threshold_offset = 0.2  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
min_speech_duration_ms = 100  # @param {type:"slider", min:50, max:1000, step:50}
min_silence_duration_ms = 700  # @param {type:"slider", min:100, max:2000, step:100}
# fmt: on

pad_sec = 0.2  # @param {type:"slider", min:0.0, max:1.0, step:0.1}

if "AUDIO_FILEPATH" in locals() and AUDIO_FILEPATH:
    print("0. Loading and Resampling Audio...")
    audio = load_and_resample_audio(
        AUDIO_FILEPATH, target_sr=TARGET_SAMPLE_RATE
    )

    # Normalize audio volume right at the start so VAD isn't starved
    original_peak = np.max(np.abs(audio))
    if original_peak > 0:
        print(f"0.1. Normalizing audio (Original peak: {original_peak:.2f})...")
        audio = audio / original_peak

    # Since this is an offline single file, prior_audio is initially None
    # so has_genuine_prior is False!
    has_genuine_prior = False

    preamble_len = int(preamble_duration_sec * TARGET_SAMPLE_RATE)
    if preamble_len > 0:
        print(
            f"0.5. Adding {preamble_duration_sec}s preamble (using low-level synthetic white noise)..."
        )
        # Generate low-level white noise to simulate a background noise floor.
        # This prevents accidentally priming the VAD with speech if the entire file is vocal.
        # We align this noise floor amplitude and seed exactly with constants!
        np.random.seed(VAD_DEFAULT_SEED)
        preamble = np.random.normal(0, VAD_NOISE_LEVEL, preamble_len).astype(
            np.float32
        )

        actual_preamble_len = len(preamble)
        extended_audio = np.concatenate([preamble, audio])
    else:
        actual_preamble_len = 0
        extended_audio = audio

    print(f"1. Applying Bandpass Filter ({highpass_hz}Hz - {lowpass_hz}Hz)...")
    bp_board = Pedalboard(
        [
            HighpassFilter(cutoff_frequency_hz=highpass_hz),
            LowpassFilter(cutoff_frequency_hz=lowpass_hz),
        ]
    )
    bp_audio = bp_board(extended_audio.astype(np.float32), TARGET_SAMPLE_RATE)

    # Apply Dynamic Range Compression (AGC) on vocal band if enabled
    if use_compressor == "Auto":
        # Use the original peak to determine if the audio was quiet enough to need compression
        # threshold aligned with constants
        should_compress = original_peak < comp_peak_threshold
        reason = f"Auto-Gated {'ON' if should_compress else 'OFF'} (Original Peak: {original_peak:.2f})"
    else:
        should_compress = use_compressor == "True"
        reason = f"Forced {use_compressor}"

    if should_compress:
        print(
            f"1.5. Applying Dynamic AGC Compressor ({reason}, Threshold: {comp_threshold_db}dB, Ratio: {comp_ratio}:1)..."
        )
        comp_board = Pedalboard(
            [
                Compressor(
                    threshold_db=comp_threshold_db,
                    ratio=comp_ratio,
                    attack_ms=comp_attack_ms,
                    release_ms=comp_release_ms,
                )
            ]
        )
        comp_audio = comp_board(bp_audio.astype(np.float32), TARGET_SAMPLE_RATE)
    else:
        print(f"1.5. Bypassing Dynamic AGC Compressor ({reason})...")
        comp_audio = bp_audio

    print(f"2. Running UL-UNAS Denoiser ({ULUNAS_MODEL_FILENAME})...")
    try:
        ulunas_denoised = run_ulunas_denoiser(comp_audio, ULUNAS_MODEL_FILENAME)

        print(
            f"3. Applying Dry/Wet Mix ({blend_ratio * 100:.0f}% Denoised, {(1 - blend_ratio) * 100:.0f}% Bandpassed)..."
        )
        mixed_audio = (
            1.0 - blend_ratio
        ) * comp_audio + blend_ratio * ulunas_denoised

        print(
            f"4. Applying Presence Boost ({boost_gain_db}dB @ {boost_freq_hz}Hz) to Mixed Audio..."
        )
        eq_board = Pedalboard(
            [
                PeakFilter(
                    cutoff_frequency_hz=boost_freq_hz,
                    gain_db=boost_gain_db,
                    q=peak_filter_q,
                )
            ]
        )
        final_audio_extended = eq_board(
            mixed_audio.astype(np.float32), TARGET_SAMPLE_RATE
        )

        # Clip to prevent IPython Audio error and simulate real hardware clipping
        final_audio_extended = np.clip(final_audio_extended, -1.0, 1.0)

        # Slice off the preamble tail before VAD inference to prevent RNN bleed-through
        # and maintain maximum VAD onset sensitivity!
        final_audio = (
            final_audio_extended[actual_preamble_len:]
            if actual_preamble_len > 0
            else final_audio_extended
        )

        # Run VAD strictly on the denoised actual audio
        final_audio_vad = final_audio
        vad_offset_sec = 0.0

        # Run Silero VAD
        print(f"5. Running Silero VAD on Denoised Audio...")
        vad_session = ort.InferenceSession(SILERO_MODEL_FILENAME)
        sr_tensor = np.array([TARGET_SAMPLE_RATE], dtype=np.int64)

        raw_final_segments = extract_vad_segments(
            final_audio_vad,
            vad_session,
            sr_tensor,
            target_sr=TARGET_SAMPLE_RATE,
            chunk_size=VAD_CHUNK_SIZE,
            context_size=VAD_CONTEXT_SIZE,
            threshold=None,
            threshold_onset=vad_threshold_onset,
            threshold_offset=vad_threshold_offset,
            min_speech_duration_ms=min_speech_duration_ms,
            min_silence_duration_ms=min_silence_duration_ms,
        )

        # Shift time coordinates back by vad_offset_sec (0.0) so they map perfectly!
        shifted_segments = []
        for start, end in raw_final_segments:
            start_sec = max(0.0, start - vad_offset_sec)
            end_sec = end - vad_offset_sec
            if end_sec > 0:
                shifted_segments.append((start_sec, end_sec))

        print(f"6. Padding segments (pad={pad_sec}s) and merging...")
        audio_len_sec = len(final_audio) / TARGET_SAMPLE_RATE
        final_segments = pad_and_merge_segments(
            shifted_segments, audio_len_sec, pad_sec=pad_sec
        )

        print(f"\nFinished! Found {len(final_segments)} speech segments.")

        print("\nPlayback:")
        # normalize=False shows the true playback volume reflecting what wavesurfer and VAD sees
        display(Audio(final_audio, rate=TARGET_SAMPLE_RATE, normalize=False))

        print("\nVisualizing results...")
        visualize_audio_with_segments(
            audio_array=final_audio,
            segments=final_segments,
            target_sr=TARGET_SAMPLE_RATE,
            title_text=f"UL-UNAS Pipeline + Hysteresis (Onset: {vad_threshold_onset}, Offset: {vad_threshold_offset}, Pad: {pad_sec}s)",
            region_color="rgba(138, 43, 226, 0.4)",
        )
    except Exception as e:
        print(f"Failed to run UL-UNAS pipeline: {e}")
else:
    print("Please upload an audio file first.")

In [ ]:
# @title VAD confidence plot

print("Calculating VAD probabilities over time...")
vad_session = ort.InferenceSession(SILERO_MODEL_FILENAME)
sr_tensor = np.array([TARGET_SAMPLE_RATE], dtype=np.int64)

state = np.zeros((2, 1, VAD_STATE_SIZE), dtype=np.float32)
context = np.zeros(VAD_CONTEXT_SIZE, dtype=np.float32)

probabilities = []
times = []

# Run VAD on the extended (primed) audio to match the pipeline
for i in range(0, len(final_audio_vad), VAD_CHUNK_SIZE):
    chunk = final_audio_vad[i : i + VAD_CHUNK_SIZE]
    if len(chunk) < VAD_CHUNK_SIZE:
        chunk = np.pad(chunk, (0, VAD_CHUNK_SIZE - len(chunk)))

    x_with_context = np.concatenate([context, chunk])
    ort_inputs = {
        "input": x_with_context.reshape(
            1, VAD_CHUNK_SIZE + VAD_CONTEXT_SIZE
        ).astype(np.float32),
        "state": state,
        "sr": sr_tensor,
    }

    outputs = vad_session.run(None, ort_inputs)
    prob = float(outputs[0].flatten()[0])
    state = outputs[1]
    context = x_with_context[-VAD_CONTEXT_SIZE:]

    probabilities.append(prob)
    # Shift the time axis back so it aligns with the original un-padded audio
    times.append((i / TARGET_SAMPLE_RATE) - vad_offset_sec)

# Plot the results
plt.figure(figsize=(15, 4))
plt.plot(times, probabilities, label="VAD Probability", color="purple")
plt.axhline(
    y=vad_threshold_offset,
    color="r",
    linestyle="--",
    label=f"Offset Threshold ({vad_threshold_offset})",
)
plt.axhline(
    y=vad_threshold_onset,
    color="g",
    linestyle="--",
    label=f"Onset Threshold ({vad_threshold_onset})",
)

# Also plot a simplified waveform behind it
time_axis = np.linspace(
    0, len(final_audio) / TARGET_SAMPLE_RATE, num=len(final_audio)
)
plt.plot(
    time_axis,
    np.abs(final_audio) / np.max(np.abs(final_audio)) * 0.5,
    color="gray",
    alpha=0.3,
    label="Audio Envelope",
)

# Restrict the view to just the true audio (hide the preamble time)
plt.xlim(0, len(final_audio) / TARGET_SAMPLE_RATE)

plt.title("Silero VAD Confidence vs Time (Primed)")
plt.xlabel("Time (seconds)")
plt.ylabel("Probability")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# @title VAD Pipeline vs. Ground Truth Comparison

file_name = pathlib.Path(AUDIO_FILEPATH).name
if file_name not in GROUND_TRUTH_SEGMENTS:
    print(
        f"No ground truth labels found for '{file_name}'. Skipping evaluation."
    )
else:
    labeled_segments = GROUND_TRUTH_SEGMENTS[file_name]

    if "final_segments" in locals() and "labeled_segments" in locals():
        evaluate_vad_performance(labeled_segments, final_segments)

        if "final_audio" in locals():
            audio_len_sec = len(final_audio) / TARGET_SAMPLE_RATE
            evaluate_vad_frame_based(
                labeled_segments, final_segments, audio_len_sec
            )

            print(
                "Visualizing Comparison (Green = Ground Truth, Purple = Detected)..."
            )
            visualize_comparison_with_segments(
                final_audio, labeled_segments, final_segments
            )
    else:
        print(
            "Make sure both 'final_segments' and 'labeled_segments' are defined."
        )